In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path

# Locate the project folder

project_folder = Path.cwd()

print("Current working directory:")
print(project_folder)

Current working directory:
c:\Users\admin\Downloads\ibank_dataset\Meridian-Analytics


In [3]:
# find all csv files in the current project folder

csv_files = list(project_folder.glob("*.csv"))
print(f"CSV files found: {len(csv_files)}")


CSV files found: 26


In [ ]:
for filr in csv_files:
    print(f"File: {filr.name}")

In [ ]:
import pandas as pd

# Read first csv file into a DataFrame
first_file = csv_files[0]
print(first_file)
df = pd.read_csv(first_file)
df.head()

In [ ]:

# Create a Data Profiling Summary

summary = []

for file in csv_files:

    # Skip documentation file
    if file.name == "data_dictionary.csv":
        continue

    # Read CSV
    df = pd.read_csv(file)

    # Identify Table Type
    if "dim" in file.stem.lower():
        table_type = "Dimension"
    elif "fact" in file.stem.lower():
        table_type = "Fact"
    else:
        table_type = "Unknown"

    # Store profiling information
    summary.append({
        "Table Name": file.stem,
        "Table Type": table_type,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

# Convert to DataFrame
profiling_report = pd.DataFrame(summary)

# Display Report
profiling_report

In [13]:
# Create an empty list to store data quality information
data_quality_report = []

# Loop through all CSV files
for file in csv_files:

    # Skip the data dictionary file
    if file.name == "data_dictionary.csv":
        continue

    # Read the current CSV file
    df = pd.read_csv(file)

    # Calculate missing values for each column
    missing_counts = df.isnull().sum()

    # Calculate duplicate rows for the current table
    duplicate_rows = df.duplicated().sum()

    # Loop through every column in the current table
    for column in df.columns:

        # Store only columns that contain missing values
        if missing_counts[column] > 0:

            # Append the result to the report
            data_quality_report.append({

                # Current table name
                "Table Name": file.stem,

                # Current column name
                "Column Name": column,

                # Total missing values
                "Missing Values": missing_counts[column],

                # Percentage of missing values
                "Missing (%)": round(
                    (missing_counts[column] / len(df)) * 100,
                    2
                ),

                # Total duplicate rows in the table
                "Duplicate Rows": duplicate_rows,

                # Percentage of duplicate rows
                "Duplicate (%)": round(
                    (duplicate_rows / len(df)) * 100,
                    2
                )
            })

# Convert the report into a DataFrame
data_quality_report = pd.DataFrame(data_quality_report)

# Display the final report
data_quality_report

,Table Name,Column Name,Missing Values,Missing (%),Duplicate Rows,Duplicate (%)
0,01_dim_clients,kyc_status,156,3.00,0,0.0
1,05_dim_instruments,maturity_date,6714,53.71,0,0.0
2,11_fact_trades,price,3505,0.50,700,0.1
3,12_fact_trade_settlements,fail_reason,679506,96.98,0,0.0
4,21_fact_credit_exposures,netting_agreement,22229,14.82,0,0.0


In [14]:
# Create an empty dictionary to store duplicate records
duplicate_tables = {}

# Loop through all CSV files
for file in csv_files:

    # Skip the data dictionary file
    if file.name == "data_dictionary.csv":
        continue

    # Read the current CSV file
    df = pd.read_csv(file)

    # Extract duplicate rows
    duplicate_df = df[df.duplicated(keep=False)]

    # Store only tables that contain duplicates
    if not duplicate_df.empty:
        duplicate_tables[file.stem] = duplicate_df

# Display duplicate tables
for table_name, duplicate_df in duplicate_tables.items():

    print("=" * 70)
    print(f"Table : {table_name}")
    print(f"Duplicate Rows : {len(duplicate_df)}")

    display(duplicate_df.head())

Table : 11_fact_trades
Duplicate Rows : 1400


,trade_id,trade_date,settlement_date,account_id,instrument_id,trader_id,counterparty_id,desk_id,side,quantity,price,currency,notional_usd,trade_status
2077,TRD000002078,2024-08-22,2024-08-25,ACC703657,INS100330,EMP10162,CPTY22108,DSK2005,Buy,450807,36.3367,TWD,16380838.72,Confirmed
2628,TRD000002629,2026-06-22,2026-06-25,ACC708941,INS111806,EMP10376,CPTY20466,DSK2007,Sell,347332,32.9824,AUD,11455842.96,Confirmed
3114,TRD000003115,2024-01-08,2024-01-09,ACC713445,INS110945,EMP10416,CPTY21894,DSK2000,Sell,70916,14.8397,NOK,1052372.17,Executed
3360,TRD000003361,2025-05-14,2025-05-17,ACC700586,INS102257,EMP10438,CPTY22477,DSK2007,Buy,424443,6.7963,TWD,2884641.96,Executed
3552,TRD000003553,2024-11-01,2024-11-01,ACC720773,INS103570,EMP10708,CPTY20484,DSK2014,Sell,157611,350.6052,NOK,55259236.18,Executed


In [15]:
# Create an empty dictionary to store rows with missing values
missing_tables = {}

# Loop through all CSV files
for file in csv_files:

    # Skip the data dictionary file
    if file.name == "data_dictionary.csv":
        continue

    # Read the current CSV file
    df = pd.read_csv(file)

    # Extract rows containing missing values
    missing_df = df[df.isnull().any(axis=1)]

    # Store only tables containing missing values
    if not missing_df.empty:
        missing_tables[file.stem] = missing_df

# Display missing value rows
for table_name, missing_df in missing_tables.items():

    print("=" * 70)
    print(f"Table : {table_name}")
    print(f"Rows with Missing Values : {len(missing_df)}")

    display(missing_df.head())

Table : 01_dim_clients
Rows with Missing Values : 156


,client_id,client_name,client_type,country,region,kyc_status,relationship_manager_id,onboarding_date,risk_rating,aum_usd
23,CLI300023,Omar O'Brien Fund,HNI,Hong Kong,APAC,NaN,EMP10559,2017-05-07,Low,11596772.11
60,CLI300060,Mei Kumar Ltd,Corporate,China,APAC,NaN,EMP10581,2026-01-06,Low,442030.42
68,CLI300068,John Fischer,HNI,France,EMEA,NaN,EMP10655,2011-03-05,Low,280463.33
90,CLI300090,John Patel Family Office,Corporate,Brazil,AMERICAS,NaN,EMP10398,2008-09-28,Medium,508323.95
126,CLI300126,Sophie Rossi Foundation,Institutional,South Korea,APAC,NaN,EMP10011,2021-01-06,Low,14574925.83


Table : 05_dim_instruments
Rows with Missing Values : 6714


,instrument_id,ticker,instrument_type,issuer_name,sector,currency,exchange,issue_date,maturity_date
1,INS100001,IRRL,FX Spot,Arjun Patel,Consumer Discretionary,AUD,LSE,2017-03-27,NaN
2,INS100002,SDKZ,FX Spot,Arjun Ivanov,Healthcare,EGP,NYSE,2003-11-27,NaN
5,INS100005,NDUI,Equity,Carlos Cohen,Financials,NOK,NASDAQ,2018-04-25,NaN
8,INS100008,WFLV,Commodity,Aisha Nguyen,Telecom,QAR,NASDAQ,2013-04-01,NaN
9,INS100009,SXKE,Equity,Elena Osei,Utilities,TWD,LSE,2002-07-03,NaN


Table : 11_fact_trades
Rows with Missing Values : 3505


,trade_id,trade_date,settlement_date,account_id,instrument_id,trader_id,counterparty_id,desk_id,side,quantity,price,currency,notional_usd,trade_status
698,TRD000000699,2024-11-09,2024-11-09,ACC722608,INS106789,EMP10063,CPTY21367,DSK2004,Buy,308864,NaN,DKK,8950353.65,Confirmed
747,TRD000000748,2025-07-04,2025-07-06,ACC720760,INS107409,EMP10739,CPTY22265,DSK2003,Sell,240139,NaN,SEK,4095594.66,Confirmed
957,TRD000000958,2023-12-23,2023-12-25,ACC719488,INS104240,EMP10789,CPTY20271,DSK2007,Sell,122065,NaN,ILS,5159748.58,Executed
998,TRD000000999,2026-03-12,2026-03-15,ACC701746,INS106896,EMP10808,CPTY20071,DSK2003,Buy,266203,NaN,KRW,8719479.26,Confirmed
1073,TRD000001074,2024-04-24,2024-04-25,ACC709236,INS101625,EMP10514,CPTY20529,DSK2025,Sell,230263,NaN,AUD,6623123.75,Confirmed


Table : 12_fact_trade_settlements
Rows with Missing Values : 679506


,settlement_id,trade_id,settlement_date,settlement_status,settlement_amount_usd,fail_reason
0,STL000000001,TRD000000001,2024-01-07,Settled,2630706.80,NaN
1,STL000000002,TRD000000002,2023-08-19,Settled,48832229.13,NaN
2,STL000000003,TRD000000003,2026-01-22,Settled,5175155.00,NaN
3,STL000000004,TRD000000004,2025-01-13,Settled,3527959.35,NaN
4,STL000000005,TRD000000005,2024-02-11,Settled,1206291.72,NaN


Table : 21_fact_credit_exposures
Rows with Missing Values : 22229


,exposure_id,counterparty_id,exposure_date,exposure_amount_usd,exposure_type,netting_agreement
3,EXP00000004,CPTY20386,2026-03-23,32993.43,Settlement Risk,NaN
4,EXP00000005,CPTY21814,2025-09-20,2235.50,Current Exposure,NaN
6,EXP00000007,CPTY21188,2024-05-18,4268.20,Potential Future Exposure,NaN
11,EXP00000012,CPTY21059,2025-06-04,13775.81,Potential Future Exposure,NaN
14,EXP00000015,CPTY20974,2024-12-09,51354.14,Potential Future Exposure,NaN


In [16]:
# Create an empty list
quality_classification = []

# Loop through every CSV file
for file in csv_files:

    # Skip data dictionary
    if file.name == "data_dictionary.csv":
        continue

    # Read current table
    df = pd.read_csv(file)

    # Missing values per column
    missing = df.isnull().sum()

    # Loop through every column
    for column in df.columns:

        if missing[column] > 0:

            quality_classification.append({

                "Table Name": file.stem,
                "Column Name": column,
                "Missing Values": missing[column],
                "Missing (%)": round((missing[column]/len(df))*100,2),})

# Convert to DataFrame
quality_classification = pd.DataFrame(quality_classification)

# Display report
quality_classification

,Table Name,Column Name,Missing Values,Missing (%)
0,01_dim_clients,kyc_status,156,3.00
1,05_dim_instruments,maturity_date,6714,53.71
2,11_fact_trades,price,3505,0.50
3,12_fact_trade_settlements,fail_reason,679506,96.98
4,21_fact_credit_exposures,netting_agreement,22229,14.82


In [17]:
# Create Staging Folder


staging_path = project_folder / "staging"

staging_path.mkdir(exist_ok=True)

print(f"Staging Folder Created : {staging_path}")

Staging Folder Created : c:\Users\admin\Downloads\ibank_dataset\Meridian-Analytics\staging


In [18]:

# Create Staging Dataset

for file in csv_files:

    # Skip data dictionary
    if file.name == "data_dictionary.csv":
        continue

    df = pd.read_csv(file)

    # -------------------------------
    # KYC Status Flag
    # -------------------------------
    if "kyc_status" in df.columns:

        df["kyc_status_flag"] = np.where(
            df["kyc_status"].isna(),
            "Missing - Needs Review",
            "OK"
        )

    # -------------------------------
    # Trade Price Flag
    # -------------------------------
    if "price" in df.columns:

        df["price_data_quality"] = np.where(
            df["price"].isna(),
            "Missing",
            "OK"
        )

    # Save into staging folder
    df.to_csv(staging_path / file.name, index=False)

print("✅ Staging Dataset Created Successfully")

✅ Staging Dataset Created Successfully


In [ ]:
# verify staging dataset creation
list(staging_path.glob("*.csv"))

In [ ]:

# Verify KYC Flag


staging_clients = pd.read_csv(staging_path / "01_dim_clients.csv")

staging_clients[
    ["client_id", "kyc_status", "kyc_status_flag"]
].head(20)

In [ ]:

# Verify Price Flag


staging_trades = pd.read_csv(staging_path / "11_fact_trades.csv")

staging_trades[
    ["trade_id", "price", "price_data_quality"]
].head(20)

In [22]:
# Read staging trade data

staging_trades = pd.read_csv(staging_path / "11_fact_trades.csv")

# Find all duplicate rows
exact_duplicates = staging_trades[staging_trades.duplicated(keep=False)]

print(f"Exact Duplicate Rows: {len(exact_duplicates)}")

exact_duplicates.head(20)

Exact Duplicate Rows: 1400


,trade_id,trade_date,settlement_date,account_id,instrument_id,trader_id,counterparty_id,desk_id,side,quantity,price,currency,notional_usd,trade_status,price_data_quality
2077,TRD000002078,2024-08-22,2024-08-25,ACC703657,INS100330,EMP10162,CPTY22108,DSK2005,Buy,450807,36.3367,TWD,16380838.72,Confirmed,OK
2628,TRD000002629,2026-06-22,2026-06-25,ACC708941,INS111806,EMP10376,CPTY20466,DSK2007,Sell,347332,32.9824,AUD,11455842.96,Confirmed,OK
3114,TRD000003115,2024-01-08,2024-01-09,ACC713445,INS110945,EMP10416,CPTY21894,DSK2000,Sell,70916,14.8397,NOK,1052372.17,Executed,OK
3360,TRD000003361,2025-05-14,2025-05-17,ACC700586,INS102257,EMP10438,CPTY22477,DSK2007,Buy,424443,6.7963,TWD,2884641.96,Executed,OK
3552,TRD000003553,2024-11-01,2024-11-01,ACC720773,INS103570,EMP10708,CPTY20484,DSK2014,Sell,157611,350.6052,NOK,55259236.18,Executed,OK
3845,TRD000003846,2023-01-29,2023-02-01,ACC713973,INS100854,EMP10040,CPTY22528,DSK2042,Sell,10172,21.6079,EGP,219795.56,Executed,OK
4415,TRD000004416,2023-12-24,2023-12-27,ACC718004,INS104383,EMP10689,CPTY21507,DSK2021,Buy,66580,11.7929,IDR,785171.28,Executed,OK
5958,TRD000005959,2023-05-10,2023-05-10,ACC705967,INS104467,EMP10165,CPTY21022,DSK2038,Sell,167738,6.0761,CHF,1019192.86,Executed,OK
6935,TRD000006936,2023-10-09,2023-10-09,ACC706529,INS107948,EMP10549,CPTY20519,DSK2011,Buy,104374,13.8745,CHF,1448137.06,Executed,OK
10598,TRD000010599,2024-11-21,2024-11-21,ACC719089,INS111431,EMP10745,CPTY21498,DSK2036,Buy,370595,13.6192,CAD,5047207.42,Executed,OK


In [ ]:
# Read only the original trade table from staging

staging_trades = pd.read_csv(staging_path / "11_fact_trades.csv")

# Find duplicate Trade IDs
trade_id_duplicates = staging_trades[
    staging_trades.duplicated(subset=["trade_id"], keep=False)
]

# Sort by Trade ID for easy comparison
trade_id_duplicates = trade_id_duplicates.sort_values("trade_id")

print(f"Rows with Duplicate Trade IDs: {len(trade_id_duplicates)}")

trade_id_duplicates.head(20)

In [24]:
# Create duplicate change log

duplicate_change_log = trade_id_duplicates.copy()

# Save duplicate change log
duplicate_change_log.to_csv(
    staging_path / "duplicate_change_log.csv",
    index=False
)

print("Duplicate Change Log Saved Successfully")

Duplicate Change Log Saved Successfully


In [25]:
# Remove exact duplicate rows

clean_trades = staging_trades.drop_duplicates()

print(f"Original Rows : {len(staging_trades)}")
print(f"Rows After Cleaning : {len(clean_trades)}")
print(f"Duplicates Removed : {len(staging_trades)-len(clean_trades)}")

Original Rows : 700700
Rows After Cleaning : 700000
Duplicates Removed : 700


In [26]:
# Save cleaned trade table

clean_trades.to_csv(
    staging_path / "11_fact_trades_clean.csv",
    index=False
)

print("Clean Trade Table Saved Successfully")

Clean Trade Table Saved Successfully


In [30]:
# Display all column names

staging_trades.columns.tolist()

['trade_id',
 'trade_date',
 'settlement_date',
 'account_id',
 'instrument_id',
 'trader_id',
 'counterparty_id',
 'desk_id',
 'side',
 'quantity',
 'price',
 'currency',
 'notional_usd',
 'trade_status',
 'price_data_quality']

In [ ]:
# Find trades with missing prices

missing_price = staging_trades[staging_trades["price"].isna()]

# Missing prices by Desk

missing_by_desk = (
    missing_price.groupby("desk_id")
    .size()
    .reset_index(name="Missing Price Count")
    .sort_values("Missing Price Count", ascending=False)
)

missing_by_desk

In [ ]:
# Missing prices by Instrument

missing_by_instrument = (
    missing_price.groupby("instrument_id")
    .size()
    .reset_index(name="Missing Price Count")
    .sort_values("Missing Price Count", ascending=False)
)

missing_by_instrument

In [29]:
# Convert trade_date into datetime

missing_price["trade_date"] = pd.to_datetime(missing_price["trade_date"])

# Missing prices by Trade Date

missing_by_date = (
    missing_price.groupby("trade_date")
    .size()
    .reset_index(name="Missing Price Count")
    .sort_values("trade_date")
)

missing_by_date

,trade_date,Missing Price Count
0,2023-01-02,6
1,2023-01-03,2
2,2023-01-04,4
3,2023-01-05,3
4,2023-01-06,3
...,...,...
1200,2026-07-03,5
1201,2026-07-05,1
1202,2026-07-06,4
1203,2026-07-07,6


In [31]:
import pandas as pd

df = pd.read_csv("01_dim_clients.csv")

print(df.columns.tolist())

['client_id', 'client_name', 'client_type', 'country', 'region', 'kyc_status', 'relationship_manager_id', 'onboarding_date', 'risk_rating', 'aum_usd']


In [32]:
import sqlalchemy
import psycopg2

print("Connection Libraries Ready ✅")

Connection Libraries Ready ✅


In [ ]:
from sqlalchemy import create_engine

# PostgreSQL Connection

username = "your_username"  # Replace with your actual username
password = "your_password"  # Replace with your actual password
host = "localhost"
port = "your_port"  # Replace with your actual port number
database = "your_database"  # Replace with your actual database name

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

print("Database Connected Successfully ✅")

Database Connected Successfully ✅


In [39]:
# Import libraries
import pandas as pd

# Upload all staging CSV files

for file in staging_path.glob("*.csv"):

    # Skip audit file
    if file.name == "duplicate_change_log.csv":
        continue

    # Read CSV
    df = pd.read_csv(file)

    # Remove only the numeric prefix
    table_name = file.stem

    if table_name[:2].isdigit():
        table_name = table_name.split("_", 1)[1]

    # Upload to PostgreSQL
    df.to_sql(
        name=table_name,
        con=engine,
        schema="staging",
        if_exists="replace",
        index=False
    )

    print(f"Uploaded: {table_name}")

print("✅ All Staging Tables Uploaded Successfully")

Uploaded: dim_clients
Uploaded: dim_employees
Uploaded: dim_offices
Uploaded: dim_desks
Uploaded: dim_instruments
Uploaded: dim_counterparties
Uploaded: dim_currencies
Uploaded: dim_products
Uploaded: dim_accounts
Uploaded: dim_risk_limits
Uploaded: fact_trades
Uploaded: fact_trades_clean
Uploaded: fact_trade_settlements
Uploaded: fact_positions_daily
Uploaded: fact_market_prices
Uploaded: fact_pnl_daily
Uploaded: fact_var_daily
Uploaded: fact_cash_flows
Uploaded: fact_corporate_actions
Uploaded: fact_compliance_alerts
Uploaded: fact_kyc_reviews
Uploaded: fact_credit_exposures
Uploaded: fact_collateral_positions
Uploaded: fact_order_book
Uploaded: fact_audit_log
Uploaded: fact_margin_calls
✅ All Staging Tables Uploaded Successfully
